### 1. Tải dữ liệu

In [ ]:
# Import thư viện
import pandas as pd
import numpy as np

# Tải file CSV vào pandas DataFrame
df = pd.read_csv('amazon.csv')

# In ra shape của dataset (số hàng, số cột)
print("Shape của dataset (số hàng, số cột):", df.shape)
print(f"Số hàng: {df.shape[0]}")
print(f"Số cột: {df.shape[1]}")

In [ ]:
# Hiển thị 5 hàng đầu tiên
df.head()

### 2. Kiểm tra thông tin cơ bản

In [ ]:
# Sử dụng .info() để kiểm tra kiểu dữ liệu của mỗi cột
df.info()

In [ ]:
# Liệt kê tên tất cả các cột
print("Danh sách tên các cột:")
print(df.columns.tolist())

### 3. Kiểm tra giá trị thiếu

In [ ]:
# Đếm số giá trị NULL/NaN trong mỗi cột
print("Số giá trị NULL/NaN trong mỗi cột:")
print(df.isnull().sum())

In [ ]:
# Tính phần trăm dữ liệu thiếu cho mỗi cột
missing_percentage = (df.isnull().sum() / len(df)) * 100
print("Phần trăm dữ liệu thiếu cho mỗi cột:")
print(missing_percentage)

In [ ]:
# Tạo bảng tổng hợp thông tin missing values
missing_df = pd.DataFrame({
    'Số lượng NULL': df.isnull().sum(),
    'Phần trăm (%)': round((df.isnull().sum() / len(df)) * 100, 2)
})
print("Bảng tổng hợp giá trị thiếu:")
missing_df[missing_df['Số lượng NULL'] > 0]

### 4. Thống kê mô tả

In [ ]:
# Sử dụng .describe() để có cái nhìn tổng quát về dữ liệu số
df.describe()

In [ ]:
# Tính giá trị min, max, mean, median, std cho các cột số chính
# Lấy các cột số
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Các cột số trong dataset:", numeric_cols)

# Tạo bảng thống kê chi tiết
stats_df = pd.DataFrame({
    'Min': df[numeric_cols].min(),
    'Max': df[numeric_cols].max(),
    'Mean': df[numeric_cols].mean(),
    'Median': df[numeric_cols].median(),
    'Std': df[numeric_cols].std()
})
print("\nThống kê mô tả cho các cột số:")
stats_df

In [ ]:
# Kiểm tra duplicate rows
print("Số hàng trùng lặp:", df.duplicated().sum())

## CÂU 2 – Làm sạch và biến đổi dữ liệu

### 1. Xử lý Missing Values

In [ ]:
# Xác định các cột có nhiều NULL nhất
print("=== XÁC ĐỊNH CÁC CỘT CÓ NHIỀU NULL NHẤT ===\n")

# Tính số lượng và phần trăm NULL cho mỗi cột
null_counts = df.isnull().sum()
null_percentage = (df.isnull().sum() / len(df)) * 100

# Tạo DataFrame tổng hợp và sắp xếp giảm dần
null_summary = pd.DataFrame({
    'Số lượng NULL': null_counts,
    'Phần trăm (%)': round(null_percentage, 2)
}).sort_values('Số lượng NULL', ascending=False)

# Hiển thị các cột có NULL
print("Các cột có giá trị NULL (sắp xếp giảm dần):")
print(null_summary[null_summary['Số lượng NULL'] > 0])
print(f"\nTổng số cột có NULL: {(null_counts > 0).sum()}")

In [ ]:
# Xử lý các giá trị NULL

"""
GIẢI THÍCH CHIẾN LƯỢC XỬ LÝ NULL:
- Với cột 'rating_count': Nếu có NULL, sẽ fill bằng 0 (giả định sản phẩm chưa có đánh giá)
- Với cột 'rating': Nếu có NULL, sẽ fill bằng median (giá trị trung vị không bị ảnh hưởng bởi outliers)
- Với các cột text (product_name, category, about_product, etc.): Fill bằng 'Unknown' hoặc drop nếu quá nhiều
- Với cột quan trọng (product_id): Drop các hàng có NULL vì đây là key identifier
"""

print("=== XỬ LÝ CÁC GIÁ TRỊ NULL ===\n")
print(f"Số hàng trước khi xử lý: {len(df)}")

# Bước 1: Drop các hàng có NULL ở cột quan trọng (product_id)
if df['product_id'].isnull().sum() > 0:
    df = df.dropna(subset=['product_id'])
    print("Đã drop các hàng có product_id = NULL")

# Bước 2: Xử lý cột rating_count - fill với 0 (chưa có đánh giá)
if 'rating_count' in df.columns and df['rating_count'].isnull().sum() > 0:
    df['rating_count'] = df['rating_count'].fillna('0')
    print("Đã fill NULL trong 'rating_count' bằng 0")

# Bước 3: Xử lý cột rating - sẽ xử lý sau khi chuyển đổi kiểu dữ liệu

# Bước 4: Fill NULL cho các cột text với 'Unknown'
text_columns = ['product_name', 'category', 'about_product', 'user_id', 'user_name', 
                'review_id', 'review_title', 'review_content', 'img_link', 'product_link']
for col in text_columns:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col] = df[col].fillna('Unknown')
        print(f"Đã fill NULL trong '{col}' bằng 'Unknown'")

# Bước 5: Drop các hàng còn lại có NULL (nếu có)
remaining_nulls = df.isnull().sum().sum()
if remaining_nulls > 0:
    print(f"\nCòn {remaining_nulls} giá trị NULL, tiến hành drop các hàng này...")
    df = df.dropna()

print(f"\nSố hàng sau khi xử lý: {len(df)}")

In [ ]:
# Kiểm tra đảm bảo dataset không còn NULL
print("=== KIỂM TRA DATASET SAU KHI XỬ LÝ NULL ===\n")

null_after = df.isnull().sum().sum()
print(f"Tổng số giá trị NULL còn lại: {null_after}")

if null_after == 0:
    print("✓ Dataset đã sạch, không còn giá trị NULL!")
else:
    print(f"✗ Vẫn còn {null_after} giá trị NULL cần xử lý")
    print(df.isnull().sum()[df.isnull().sum() > 0])

### 2. Chuyển đổi kiểu dữ liệu

In [ ]:
# Kiểm tra kiểu dữ liệu hiện tại trước khi chuyển đổi
print("=== KIỂU DỮ LIỆU HIỆN TẠI ===\n")
print(df[['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']].dtypes)
print("\n--- Mẫu dữ liệu ---")
print(df[['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']].head())

In [ ]:
# Chuyển đổi discounted_price và actual_price: string → float
# Loại bỏ ký hiệu ₹ và dấu phẩy ngăn cách hàng nghìn

print("=== CHUYỂN ĐỔI discounted_price VÀ actual_price ===\n")

# Hàm chuyển đổi giá từ string sang float
def convert_price(price_str):
    """
    Chuyển đổi giá tiền từ string (₹1,099) sang float (1099.0)
    - Loại bỏ ký hiệu ₹
    - Loại bỏ dấu phẩy
    """
    if pd.isna(price_str):
        return np.nan
    # Loại bỏ ₹ và dấu phẩy, sau đó chuyển sang float
    price_clean = str(price_str).replace('₹', '').replace(',', '').strip()
    try:
        return float(price_clean)
    except ValueError:
        return np.nan

# Áp dụng chuyển đổi
df['discounted_price'] = df['discounted_price'].apply(convert_price)
df['actual_price'] = df['actual_price'].apply(convert_price)

print("✓ Đã chuyển đổi 'discounted_price' sang float")
print("✓ Đã chuyển đổi 'actual_price' sang float")
print(f"\nKiểu dữ liệu mới:")
print(f"  - discounted_price: {df['discounted_price'].dtype}")
print(f"  - actual_price: {df['actual_price'].dtype}")

In [ ]:
# Chuyển đổi discount_percentage: loại bỏ % → float

print("=== CHUYỂN ĐỔI discount_percentage ===\n")

def convert_percentage(pct_str):
    """
    Chuyển đổi phần trăm từ string (64%) sang float (64.0)
    - Loại bỏ ký hiệu %
    """
    if pd.isna(pct_str):
        return np.nan
    # Loại bỏ % và chuyển sang float
    pct_clean = str(pct_str).replace('%', '').strip()
    try:
        return float(pct_clean)
    except ValueError:
        return np.nan

df['discount_percentage'] = df['discount_percentage'].apply(convert_percentage)

print("✓ Đã chuyển đổi 'discount_percentage' sang float (đã loại bỏ %)")
print(f"Kiểu dữ liệu mới: {df['discount_percentage'].dtype}")
print(f"\nGiá trị mẫu: {df['discount_percentage'].head().tolist()}")

In [ ]:
# Chuyển đổi rating: đảm bảo là float, xử lý giá trị không hợp lệ

print("=== CHUYỂN ĐỔI rating ===\n")

def convert_rating(rating_str):
    """
    Chuyển đổi rating sang float
    - Xử lý các giá trị không hợp lệ (ví dụ: '|', text, etc.)
    - Rating hợp lệ: 0.0 - 5.0
    """
    if pd.isna(rating_str):
        return np.nan
    try:
        rating = float(rating_str)
        # Kiểm tra rating có nằm trong khoảng hợp lệ (0-5)
        if 0 <= rating <= 5:
            return rating
        else:
            return np.nan  # Giá trị ngoài khoảng hợp lệ
    except (ValueError, TypeError):
        return np.nan  # Không thể chuyển đổi

# Áp dụng chuyển đổi
df['rating'] = df['rating'].apply(convert_rating)

# Fill các giá trị NULL (nếu có) bằng median của rating
rating_median = df['rating'].median()
df['rating'] = df['rating'].fillna(rating_median)

print("✓ Đã chuyển đổi 'rating' sang float")
print(f"✓ Đã fill các giá trị không hợp lệ bằng median = {rating_median}")
print(f"Kiểu dữ liệu mới: {df['rating'].dtype}")
print(f"\nThống kê rating:")
print(df['rating'].describe())

In [ ]:
# Chuyển đổi rating_count: string → int

print("=== CHUYỂN ĐỔI rating_count ===\n")

def convert_rating_count(count_str):
    """
    Chuyển đổi rating_count từ string sang int
    - Loại bỏ dấu phẩy ngăn cách hàng nghìn (ví dụ: "24,269" → 24269)
    """
    if pd.isna(count_str):
        return 0
    # Loại bỏ dấu phẩy và chuyển sang int
    count_clean = str(count_str).replace(',', '').strip()
    try:
        return int(float(count_clean))  # Dùng float trước để xử lý các trường hợp như "24269.0"
    except ValueError:
        return 0

df['rating_count'] = df['rating_count'].apply(convert_rating_count)

print("✓ Đã chuyển đổi 'rating_count' sang int")
print(f"Kiểu dữ liệu mới: {df['rating_count'].dtype}")
print(f"\nGiá trị mẫu: {df['rating_count'].head().tolist()}")

### 3. Tạo cột mới

In [ ]:
# Tạo các cột mới

print("=== TẠO CÁC CỘT MỚI ===\n")

# Tỷ giá: 1 USD = 83 INR
USD_TO_INR = 83

# 1. discounted_price_usd: Giá đã giảm tính bằng USD
df['discounted_price_usd'] = round(df['discounted_price'] / USD_TO_INR, 2)
print("✓ Đã tạo cột 'discounted_price_usd' (1 USD = 83 INR)")

# 2. actual_price_usd: Giá gốc tính bằng USD
df['actual_price_usd'] = round(df['actual_price'] / USD_TO_INR, 2)
print("✓ Đã tạo cột 'actual_price_usd'")

# 3. discount_amount: Số tiền giảm giá = actual_price - discounted_price
df['discount_amount'] = df['actual_price'] - df['discounted_price']
print("✓ Đã tạo cột 'discount_amount' = actual_price - discounted_price")

# 4. price_ratio: Tỷ lệ giá = discounted_price / actual_price
df['price_ratio'] = round(df['discounted_price'] / df['actual_price'], 4)
print("✓ Đã tạo cột 'price_ratio' = discounted_price / actual_price")

print("\n--- Các cột mới đã tạo ---")
print(df[['discounted_price', 'discounted_price_usd', 'actual_price', 'actual_price_usd', 
          'discount_amount', 'price_ratio']].head())

### 4. Hiển thị dữ liệu sau khi xử lý

In [ ]:
# Hiển thị 5 dòng sau khi xử lý

print("=== 5 DÒNG ĐẦU TIÊN SAU KHI XỬ LÝ ===\n")

# Hiển thị các cột quan trọng đã xử lý
display_cols = ['product_id', 'product_name', 'discounted_price', 'actual_price', 
                'discount_percentage', 'rating', 'rating_count',
                'discounted_price_usd', 'actual_price_usd', 'discount_amount', 'price_ratio']

df[display_cols].head()

In [ ]:
# Kiểm tra kiểu dữ liệu cuối cùng và tổng kết

print("=== TỔNG KẾT SAU KHI XỬ LÝ DỮ LIỆU ===\n")

print("1. Kiểu dữ liệu các cột đã chuyển đổi:")
print(df[['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']].dtypes)

print("\n2. Các cột mới đã tạo:")
print(df[['discounted_price_usd', 'actual_price_usd', 'discount_amount', 'price_ratio']].dtypes)

print(f"\n3. Số lượng NULL còn lại: {df.isnull().sum().sum()}")
print(f"4. Shape của dataset: {df.shape}")

print("\n5. Thông tin chi tiết:")
df.info()

## CÂU 3 – Phân tích thống kê về giá

### 1. Thống kê discounted_price_usd

In [ ]:
price = df['discounted_price_usd']

mean_val = price.mean()
median_val = price.median()
mode_val = price.mode()[0]

std_val = price.std()
var_val = price.var()

Q1 = price.quantile(0.25)
Q2 = price.quantile(0.50)
Q3 = price.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(price < lower_bound) | (price > upper_bound)]

print(f"Mean: {mean_val:.2f}")
print(f"Median: {median_val:.2f}")
print(f"Mode: {mode_val:.2f}")
print(f"\nStd: {std_val:.2f}")
print(f"Variance: {var_val:.2f}")
print(f"\nQ1: {Q1:.2f}, Q2: {Q2:.2f}, Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"\nOutlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Số lượng sản phẩm là outlier: {len(outliers)}")

### 2. Phân tích chiết khấu

In [ ]:
avg_discount = df['discount_percentage'].mean()
print(f"Discount percentage trung bình: {avg_discount:.2f}%")

max_discount_idx = df['discount_percentage'].idxmax()
min_discount_idx = df['discount_percentage'].idxmin()

print(f"\nSản phẩm chiết khấu cao nhất ({df.loc[max_discount_idx, 'discount_percentage']}%):")
print(f"  - {df.loc[max_discount_idx, 'product_name'][:80]}...")

print(f"\nSản phẩm chiết khấu thấp nhất ({df.loc[min_discount_idx, 'discount_percentage']}%):")
print(f"  - {df.loc[min_discount_idx, 'product_name'][:80]}...")

In [ ]:
bins = [0, 25, 50, 75, 100]
labels = ['0-25%', '25-50%', '50-75%', '75-100%']
df['discount_group'] = pd.cut(df['discount_percentage'], bins=bins, labels=labels, include_lowest=True)

group_counts = df['discount_group'].value_counts().sort_index()
print("Số sản phẩm theo nhóm chiết khấu:")
print(group_counts)

### 3. Tương quan

In [ ]:
corr_price = np.corrcoef(df['actual_price_usd'], df['discounted_price_usd'])[0, 1]
corr_discount_rating = np.corrcoef(df['discount_percentage'], df['rating'])[0, 1]

print(f"Correlation giữa actual_price_usd và discounted_price_usd: {corr_price:.4f}")
print(f"Correlation giữa discount_percentage và rating: {corr_discount_rating:.4f}")

## CÂU 4 – Phân tích đánh giá sản phẩm

### 1. Thống kê rating

In [ ]:
df['rating_level'] = df['rating'].apply(lambda x: int(x) if x == int(x) else int(x) + 1)
df['rating_level'] = df['rating'].round().astype(int)

rating_counts = df['rating_level'].value_counts().sort_index()
rating_percentage = (rating_counts / len(df) * 100).round(2)

print("Số lượng sản phẩm theo mức rating:")
print(rating_counts)
print("\nTỷ lệ phần trăm:")
for level, pct in rating_percentage.items():
    print(f"  {level} sao: {pct}%")

### 2. Phân tích lượt đánh giá

In [ ]:
from scipy.stats import skew, kurtosis

rc = df['rating_count']
skewness = skew(rc)
kurt = kurtosis(rc)

print("Phân bố của rating_count:")
print(f"  Skewness: {skewness:.2f}")
print(f"  Kurtosis: {kurt:.2f}")

if skewness > 1:
    print("  → Phân bố lệch phải (right-skewed/positive skewed)")
elif skewness < -1:
    print("  → Phân bố lệch trái (left-skewed/negative skewed)")
else:
    print("  → Phân bố gần normal")

percentiles = [25, 50, 75, 90, 95]
print("\nPercentiles của rating_count:")
for p in percentiles:
    val = np.percentile(rc, p)
    print(f"  P{p}: {val:.0f}")

max_rc_idx = df['rating_count'].idxmax()
print(f"\nSản phẩm có lượt đánh giá cao nhất:")
print(f"  - {df.loc[max_rc_idx, 'product_name'][:80]}...")
print(f"  - Rating count: {df.loc[max_rc_idx, 'rating_count']:,}")

### 3. Mối quan hệ giữa rating và lượt đánh giá

In [ ]:
corr_rating_count = np.corrcoef(df['rating'], df['rating_count'])[0, 1]
print(f"Correlation giữa rating và rating_count: {corr_rating_count:.4f}")

In [ ]:
df['main_category'] = df['category'].apply(lambda x: x.split('|')[0] if '|' in str(x) else x)

pivot_table = pd.pivot_table(df, values='product_id', index='main_category', 
                              columns='rating_level', aggfunc='count', fill_value=0)
print("Ma trận pivot: Category vs Rating (số lượng sản phẩm)")
pivot_table

### 4. Tìm sản phẩm

In [ ]:
top10_rating = df.nlargest(10, 'rating')[['product_name', 'rating', 'rating_count']]
print("Top 10 sản phẩm có rating cao nhất:")
top10_rating

In [ ]:
top10_count = df.nlargest(10, 'rating_count')[['product_name', 'rating', 'rating_count']]
print("Top 10 sản phẩm được đánh giá nhiều nhất:")
top10_count

In [ ]:
common_products = set(top10_rating.index) & set(top10_count.index)
print(f"Số sản phẩm trùng lặp giữa 2 danh sách: {len(common_products)}")

if len(common_products) > 0:
    print("\nCác sản phẩm trùng lặp:")
    for idx in common_products:
        print(f"  - {df.loc[idx, 'product_name'][:60]}...")
else:
    print("Không có sản phẩm nào trùng lặp giữa 2 danh sách.")

## CÂU 5 – Phân tích theo danh mục sản phẩm

### 1. Xử lý cột category

In [ ]:
print("Mẫu dữ liệu cột category:")
print(df['category'].head(3).tolist())

df['category_list'] = df['category'].apply(lambda x: str(x).split('|'))
df['main_category'] = df['category'].apply(lambda x: str(x).split('|')[0])
df['sub_category'] = df['category'].apply(lambda x: str(x).split('|')[1] if len(str(x).split('|')) > 1 else 'Unknown')

all_categories = []
for cat_list in df['category_list']:
    all_categories.extend(cat_list)
unique_categories = list(set(all_categories))

print(f"\nSố lượng main category: {df['main_category'].nunique()}")
print(f"Tổng số unique sub-categories: {len(unique_categories)}")
print(f"\nDanh sách main categories:")
print(df['main_category'].unique())

### 2. Thống kê sản phẩm theo danh mục

In [ ]:
category_counts = df['main_category'].value_counts()
category_percentage = (category_counts / len(df) * 100).round(2)

category_stats = pd.DataFrame({
    'Số lượng': category_counts,
    'Tỷ lệ (%)': category_percentage
})

print("Thống kê sản phẩm theo danh mục:")
category_stats

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.barh(category_counts.index, category_counts.values, color='steelblue')
plt.xlabel('Số lượng sản phẩm')
plt.ylabel('Danh mục')
plt.title('Số lượng sản phẩm theo danh mục')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 3. Phân tích theo category

In [ ]:
category_analysis = df.groupby('main_category').agg({
    'discount_amount': 'sum',
    'rating': 'mean',
    'rating_count': 'mean'
}).round(2)

category_analysis.columns = ['Tổng giá trị khuyến mãi (INR)', 'Rating trung bình', 'Lượt đánh giá TB']
category_analysis = category_analysis.sort_values('Tổng giá trị khuyến mãi (INR)', ascending=False)

print("Phân tích theo category:")
category_analysis

In [ ]:
best_rating_category = category_analysis['Rating trung bình'].idxmax()
best_rating_value = category_analysis['Rating trung bình'].max()

print(f"Category có rating tốt nhất: {best_rating_category}")
print(f"Rating trung bình: {best_rating_value}")

## CÂU 6 – Phân tích người dùng và review

### 1. Thống kê người dùng

In [ ]:
unique_users = df['user_id'].nunique()
unique_reviews = df['review_id'].nunique()
total_reviews = len(df)
avg_reviews_per_user = total_reviews / unique_users

print(f"Số lượng người dùng độc nhất: {unique_users:,}")
print(f"Số lượng review độc nhất: {unique_reviews:,}")
print(f"Số review trung bình trên mỗi người dùng: {avg_reviews_per_user:.2f}")

### 2. Phân tích hoạt động người dùng

In [ ]:
top10_users = df['user_id'].value_counts().head(10)
print("Top 10 người dùng viết nhiều review nhất:")
print(top10_users)

In [ ]:
user_products = df.groupby('user_id')['product_id'].nunique().reset_index()
user_products.columns = ['user_id', 'unique_products']
user_products = user_products.sort_values('unique_products', ascending=False)

print("Số lượng unique products mỗi người dùng đã review (top 10):")
user_products.head(10)

### 3. Phân tích độ dài review

In [ ]:
df['title_length'] = df['review_title'].astype(str).apply(len)
df['content_length'] = df['review_content'].astype(str).apply(len)

print("Thống kê độ dài review_title:")
print(df['title_length'].describe())
print("\nThống kê độ dài review_content:")
print(df['content_length'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['title_length'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Độ dài')
axes[0].set_ylabel('Tần suất')
axes[0].set_title('Phân bố độ dài Review Title')

axes[1].hist(df['content_length'], bins=30, color='coral', edgecolor='black')
axes[1].set_xlabel('Độ dài')
axes[1].set_ylabel('Tần suất')
axes[1].set_title('Phân bố độ dài Review Content')

plt.tight_layout()
plt.show()

### 4. Tương quan review và rating

In [ ]:
corr_title_rating = np.corrcoef(df['title_length'], df['rating'])[0, 1]
corr_content_rating = np.corrcoef(df['content_length'], df['rating'])[0, 1]

print(f"Correlation giữa độ dài title và rating: {corr_title_rating:.4f}")
print(f"Correlation giữa độ dài content và rating: {corr_content_rating:.4f}")

avg_length_by_rating = df.groupby('rating_level')[['title_length', 'content_length']].mean().round(2)
print("\nĐộ dài review trung bình theo mức rating:")
avg_length_by_rating

In [ ]:
from collections import Counter
import re

all_titles = ' '.join(df['review_title'].astype(str).tolist())
words = re.findall(r'\b[a-zA-Z]{3,}\b', all_titles.lower())

word_counts = Counter(words)
top_words = word_counts.most_common(20)

print("Top 20 từ xuất hiện nhiều nhất trong review_title:")
for word, count in top_words:
    print(f"  {word}: {count}")

## CÂU 7 – Trực quan hóa dữ liệu

### 1. Top 15 category theo số lượng sản phẩm

In [ ]:
import seaborn as sns

top15_cat = df['main_category'].value_counts().head(15)

plt.figure(figsize=(12, 8))
plt.barh(top15_cat.index[::-1], top15_cat.values[::-1], color='steelblue')
plt.xlabel('Số lượng sản phẩm')
plt.ylabel('Category')
plt.title('Top 15 Category theo số lượng sản phẩm')
plt.tight_layout()
plt.show()

### 2. Phân bố sản phẩm theo category (Top 10 + Others)

In [ ]:
cat_counts = df['main_category'].value_counts()
top10 = cat_counts.head(10)
others = cat_counts[10:].sum()

pie_data = pd.concat([top10, pd.Series({'Others': others})])

plt.figure(figsize=(10, 10))
plt.pie(pie_data.values, labels=pie_data.index, autopct='%1.1f%%', startangle=90)
plt.title('Phân bố sản phẩm theo Category (Top 10 + Others)')
plt.tight_layout()
plt.show()

### 3. Correlation Heatmap

In [ ]:
corr_cols = ['actual_price_usd', 'discounted_price_usd', 'discount_percentage', 'rating', 'rating_count']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.3f', linewidths=0.5)
plt.title('Correlation Heatmap giữa các biến số')
plt.tight_layout()
plt.show()

### 4. Biểu đồ kết hợp Subplots (2x2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1) Phân bố giá
axes[0, 0].hist(df['discounted_price_usd'], bins=50, color='steelblue', edgecolor='black')
axes[0, 0].set_xlabel('Giá (USD)')
axes[0, 0].set_ylabel('Tần suất')
axes[0, 0].set_title('Phân bố giá sản phẩm (discounted_price_usd)')

# 2) Phân bố rating
axes[0, 1].hist(df['rating'], bins=20, color='coral', edgecolor='black')
axes[0, 1].set_xlabel('Rating')
axes[0, 1].set_ylabel('Tần suất')
axes[0, 1].set_title('Phân bố Rating')

# 3) Category distribution
top5_cat = df['main_category'].value_counts().head(5)
axes[1, 0].bar(range(len(top5_cat)), top5_cat.values, color='green')
axes[1, 0].set_xticks(range(len(top5_cat)))
axes[1, 0].set_xticklabels(top5_cat.index, rotation=45, ha='right')
axes[1, 0].set_xlabel('Category')
axes[1, 0].set_ylabel('Số lượng')
axes[1, 0].set_title('Top 5 Category Distribution')

# 4) Price vs Rating
axes[1, 1].scatter(df['discounted_price_usd'], df['rating'], alpha=0.5, s=10, c='purple')
axes[1, 1].set_xlabel('Giá (USD)')
axes[1, 1].set_ylabel('Rating')
axes[1, 1].set_title('Price vs Rating')

plt.tight_layout()
plt.show()

## CÂU 8 – Khám phá dữ liệu văn bản

### 1. Xử lý văn bản cơ bản

In [ ]:
df_text = df.copy()

print(f"Số NULL trong review_content trước xử lý: {df_text['review_content'].isnull().sum()}")
df_text = df_text.dropna(subset=['review_content'])
print(f"Số hàng sau khi xóa NULL: {len(df_text)}")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_text['review_clean'] = df_text['review_content'].apply(clean_text)

print("\nMẫu review sau khi xử lý:")
print(df_text['review_clean'].head(3).tolist())

### 2. Thống kê từ

In [ ]:
all_words = []
for review in df_text['review_clean']:
    words = review.split()
    all_words.extend(words)

word_counts = Counter(all_words)
top30_words = word_counts.most_common(30)

print("Top 30 từ phổ biến nhất trong review_content:")
for i, (word, count) in enumerate(top30_words, 1):
    print(f"  {i}. {word}: {count}")

### 3. Phân tích độ dài

In [ ]:
df_text['word_count'] = df_text['review_clean'].apply(lambda x: len(x.split()))

print("Thống kê số từ trong mỗi review:")
print(df_text['word_count'].describe())

corr_words_rating = np.corrcoef(df_text['word_count'], df_text['rating'])[0, 1]
print(f"\nCorrelation giữa số từ và rating: {corr_words_rating:.4f}")

In [ ]:
def categorize_length(word_count):
    if word_count < 10:
        return 'very short (<10)'
    elif word_count < 50:
        return 'short (10-50)'
    elif word_count < 200:
        return 'medium (50-200)'
    else:
        return 'long (>200)'

df_text['length_group'] = df_text['word_count'].apply(categorize_length)

group_stats = df_text.groupby('length_group').agg({
    'rating': 'mean',
    'product_id': 'count'
}).round(2)
group_stats.columns = ['Rating trung bình', 'Số lượng review']

print("Thống kê theo nhóm độ dài review:")
group_stats

## CÂU 9* – Tổng hợp phân tích

### 1. Top 10 sản phẩm xuất sắc

In [ ]:
from sklearn.preprocessing import MinMaxScaler

df_analysis = df.copy()

scaler = MinMaxScaler()
df_analysis['rating_norm'] = scaler.fit_transform(df_analysis[['rating']])
df_analysis['rating_count_norm'] = scaler.fit_transform(df_analysis[['rating_count']])
df_analysis['price_norm'] = 1 - scaler.fit_transform(df_analysis[['discounted_price_usd']])

df_analysis['score'] = (
    df_analysis['rating_norm'] * 0.4 + 
    df_analysis['rating_count_norm'] * 0.35 + 
    df_analysis['price_norm'] * 0.25
)

top10_excellent = df_analysis.nlargest(10, 'score')[
    ['product_name', 'rating', 'rating_count', 'discounted_price_usd', 'score']
]
top10_excellent.columns = ['Tên sản phẩm', 'Rating', 'Lượt đánh giá', 'Giá (USD)', 'Điểm tổng hợp']

print("TOP 10 SẢN PHẨM XUẤT SẮC (Rating cao + Lượt đánh giá nhiều + Giá cạnh tranh):")
print("Công thức: Score = 0.4*Rating + 0.35*Rating_count + 0.25*(1-Price)")
top10_excellent

### 2. Bottom 10 sản phẩm kém

In [ ]:
bottom10_poor = df_analysis.nsmallest(10, 'score')[
    ['product_name', 'rating', 'rating_count', 'discounted_price_usd', 'score']
]
bottom10_poor.columns = ['Tên sản phẩm', 'Rating', 'Lượt đánh giá', 'Giá (USD)', 'Điểm tổng hợp']

print("BOTTOM 10 SẢN PHẨM KÉM (Cần cải thiện hoặc loại bỏ):")
print("Đặc điểm: Rating thấp, ít lượt đánh giá, giá cao")
bottom10_poor

### 3. Sản phẩm rating tốt nhưng giá cao

In [ ]:
rating_threshold = df['rating'].quantile(0.9)
price_threshold = df['discounted_price_usd'].quantile(0.75)

high_rating_high_price = df[
    (df['rating'] >= rating_threshold) & 
    (df['discounted_price_usd'] >= price_threshold)
].sort_values(['rating', 'discounted_price_usd'], ascending=[False, False])

print(f"Ngưỡng rating (P90): {rating_threshold:.2f}")
print(f"Ngưỡng giá (P75): ${price_threshold:.2f}")
print(f"\nSố sản phẩm rating tốt (>= {rating_threshold}) nhưng giá cao (>= ${price_threshold:.2f}): {len(high_rating_high_price)}")

result = high_rating_high_price[['product_name', 'rating', 'rating_count', 'discounted_price_usd', 'main_category']].head(15)
result.columns = ['Tên sản phẩm', 'Rating', 'Lượt đánh giá', 'Giá (USD)', 'Danh mục']
print("\nTop 15 sản phẩm rating cao nhất trong nhóm giá cao:")
result

In [ ]:
category_premium = high_rating_high_price.groupby('main_category').agg({
    'product_id': 'count',
    'rating': 'mean',
    'discounted_price_usd': 'mean'
}).round(2)
category_premium.columns = ['Số sản phẩm', 'Rating TB', 'Giá TB (USD)']
category_premium = category_premium.sort_values('Số sản phẩm', ascending=False)

print("Phân tích sản phẩm rating cao - giá cao theo danh mục:")
category_premium

## CÂU 10* – Dashboard tổng hợp kết quả

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(20, 18))
fig.suptitle('AMAZON SALES DATA - DASHBOARD TỔNG HỢP', fontsize=20, fontweight='bold', y=1.02)

# 1) Phân bố giá (Histogram)
axes[0, 0].hist(df['discounted_price_usd'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Giá (USD)')
axes[0, 0].set_ylabel('Tần suất')
axes[0, 0].set_title('1. Phân bố giá sản phẩm')
axes[0, 0].axvline(df['discounted_price_usd'].median(), color='red', linestyle='--', label=f"Median: ${df['discounted_price_usd'].median():.2f}")
axes[0, 0].legend()

# 2) Phân bố rating (Histogram)
axes[0, 1].hist(df['rating'], bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Rating')
axes[0, 1].set_ylabel('Tần suất')
axes[0, 1].set_title('2. Phân bố Rating')
axes[0, 1].axvline(df['rating'].mean(), color='blue', linestyle='--', label=f"Mean: {df['rating'].mean():.2f}")
axes[0, 1].legend()

# 3) Top 10 Categories (Bar chart)
top10_cat = df['main_category'].value_counts().head(10)
axes[0, 2].barh(top10_cat.index[::-1], top10_cat.values[::-1], color='teal')
axes[0, 2].set_xlabel('Số lượng sản phẩm')
axes[0, 2].set_title('3. Top 10 Categories')

# 4) Giá vs Rating (Scatter plot)
axes[1, 0].scatter(df['discounted_price_usd'], df['rating'], alpha=0.4, s=15, c='purple')
axes[1, 0].set_xlabel('Giá (USD)')
axes[1, 0].set_ylabel('Rating')
axes[1, 0].set_title('4. Giá vs Rating')

# 5) Discount vs Rating (Scatter plot)
axes[1, 1].scatter(df['discount_percentage'], df['rating'], alpha=0.4, s=15, c='green')
axes[1, 1].set_xlabel('Discount (%)')
axes[1, 1].set_ylabel('Rating')
axes[1, 1].set_title('5. Discount vs Rating')

# 6) Rating Heatmap by Category
pivot_rating = df.groupby('main_category')['rating'].agg(['mean', 'count']).round(2)
pivot_rating = pivot_rating.sort_values('count', ascending=False).head(10)
colors = plt.cm.RdYlGn((pivot_rating['mean'] - 3) / 2)
axes[1, 2].barh(pivot_rating.index[::-1], pivot_rating['mean'].values[::-1], color=colors[::-1])
axes[1, 2].set_xlabel('Rating trung bình')
axes[1, 2].set_title('6. Rating TB theo Category (Top 10)')
axes[1, 2].set_xlim(3, 5)

# 7) Top 10 Products by Rating Count
top10_products = df.nlargest(10, 'rating_count')
y_labels = [name[:30] + '...' if len(name) > 30 else name for name in top10_products['product_name']]
axes[2, 0].barh(y_labels[::-1], top10_products['rating_count'].values[::-1], color='orange')
axes[2, 0].set_xlabel('Số lượt đánh giá')
axes[2, 0].set_title('7. Top 10 sản phẩm được đánh giá nhiều nhất')

# 8) Review Length Distribution
axes[2, 1].hist(df['content_length'], bins=50, color='mediumpurple', edgecolor='black', alpha=0.7)
axes[2, 1].set_xlabel('Độ dài review (ký tự)')
axes[2, 1].set_ylabel('Tần suất')
axes[2, 1].set_title('8. Phân bố độ dài Review Content')

# 9) Discount Group Distribution (Pie chart)
discount_dist = df['discount_group'].value_counts().sort_index()
axes[2, 2].pie(discount_dist.values, labels=discount_dist.index, autopct='%1.1f%%', 
               colors=['#ff9999','#66b3ff','#99ff99','#ffcc99'], startangle=90)
axes[2, 2].set_title('9. Phân bố nhóm chiết khấu')

plt.tight_layout()
plt.show()